# LifeMate Vast — resume từ checkpoint 508

Notebook này chỉ làm một luồng: **kiểm tra máy → chuẩn bị SFT đúng cấu hình cũ → resume checkpoint 508 → đánh giá toàn bộ dev**. Không chạy smoke và không tạo public submission.

Trước khi chạy, cần instance có GPU tối thiểu 24 GB VRAM và ít nhất 18 GiB trống. Nếu thiếu dung lượng, notebook dừng ở bước kiểm tra đầu tiên.

## 1. Cấu hình run

In [ ]:
from pathlib import Path
import hashlib, json, os, random, shutil, subprocess, sys, time

# Tiếp tục baseline cũ; không xếp hạng tool theo transcript ASR.
os.environ["LIFEMATE_TOOL_RETRIEVAL"] = "off"

CWD = Path.cwd().resolve()
SEARCH_DIRS = [CWD, *CWD.parents, Path("/workspace/lifemate")]
PROJECT_DIR = next((p for p in SEARCH_DIRS if (p / "training/train_lora.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Không tìm thấy project LifeMate. Kernel đang ở: {CWD}")
os.chdir(PROJECT_DIR)

DATA_ROOT = PROJECT_DIR / "data/raw/train+public_test"
RUN_ID = "vast-20261106-124816"
RUN_DIR = PROJECT_DIR / "lifemate_runs" / RUN_ID
FULL_DIR = RUN_DIR / "full"
MODEL_PATH = PROJECT_DIR / "models/Qwen3-4B-Instruct-2507"
RESUME_CHECKPOINT = FULL_DIR / "checkpoint-508"

SEED = 2026
LR = 5e-5
MAX_EPOCHS = 30
MAX_SEQUENCE_LENGTH = 8192
GRAD_ACCUMULATION = 16
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVALS_PER_EPOCH = 2
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.001
SCHEMA_PERMUTATIONS = 1  # Giữ đúng dữ liệu augmentation của run cũ

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_DIR)
print("Run:", RUN_DIR)
print("Resume from:", RESUME_CHECKPOINT)


## 2. Kiểm tra GPU, dung lượng và checkpoint

Cell này không cài hay tải gì trước khi xác nhận ổ đĩa đủ chỗ.

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
free_gib = shutil.disk_usage(PROJECT_DIR).free / 1024**3
print(f"Free disk: {free_gib:.1f} GiB")
if free_gib < 18:
    raise RuntimeError("Ổ đĩa còn dưới 18 GiB. Dừng tại đây; cần dọn file thừa hoặc tạo instance có disk lớn hơn.")

try:
    import torch
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu130"])
    import torch
if not torch.cuda.is_available():
    raise RuntimeError("Python kernel không thấy CUDA GPU. Chọn kernel Python của /venv/main.")
props = torch.cuda.get_device_properties(0)
print(f"GPU={props.name}; VRAM={props.total_memory/1024**3:.1f} GiB")
if props.total_memory < 23 * 1024**3:
    raise RuntimeError("Cần GPU hạng 24 GB VRAM trở lên.")

# Dataset is archived in the bundle; extract it before checking raw files.
if not (DATA_ROOT / "train/train.jsonl").is_file():
    dataset_archive = PROJECT_DIR / "data/archives/train+public_test.zip"
    if not dataset_archive.is_file():
        raise FileNotFoundError(dataset_archive)
    (PROJECT_DIR / "data/raw").mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-n", "-q", str(dataset_archive), "train+public_test/*", "-d", str(PROJECT_DIR / "data/raw")], check=True)

required = [
    DATA_ROOT / "train/train.jsonl",
    DATA_ROOT / "train/train_tools_catalog.json",
    DATA_ROOT / "public_test/public_test.jsonl",
    RESUME_CHECKPOINT / "adapter_config.json",
    RESUME_CHECKPOINT / "trainer_state.json",
    RESUME_CHECKPOINT / "adapter_model.safetensors",
    RESUME_CHECKPOINT / "optimizer.pt",
    RESUME_CHECKPOINT / "scheduler.pt",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Thiếu file cần cho resume:\n" + "\n".join(missing))
resume_step = int(RESUME_CHECKPOINT.name.rsplit("-", 1)[1])
newer = [p for p in FULL_DIR.glob("checkpoint-*") if p.is_dir() and int(p.name.rsplit("-", 1)[1]) > resume_step]
if newer:
    raise RuntimeError("Đã có checkpoint mới hơn mốc đang chọn. Sửa RESUME_CHECKPOINT thành checkpoint mới nhất trước khi chạy lại.")
print("GPU, dataset và checkpoint: OK")


## 3. Cài thư viện và chuẩn bị dữ liệu

Chuẩn bị lại SFT với thứ tự tool cũ. Nếu model đã nằm trên đĩa, lệnh download chỉ xác minh/cập nhật file còn thiếu.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"], check=True)
subprocess.run([sys.executable, "scripts/download_models.py"], check=True)
subprocess.run([
    sys.executable, "training/prepare_sft.py",
    "--data-root", str(DATA_ROOT),
    "--output-dir", "data/processed/sft",
    "--seed", str(SEED),
], cwd=PROJECT_DIR, check=True)
manifest = json.loads((PROJECT_DIR / "data/processed/sft/manifest.json").read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 4. Tạo bản augmentation giống run cũ

In [ ]:
TRAIN_JSONL = PROJECT_DIR / "data/processed/sft/train.jsonl"
DEV_JSONL = PROJECT_DIR / "data/processed/sft/dev.jsonl"
AUG_TRAIN_JSONL = RUN_DIR / "train_schema_augmented.jsonl"
if SCHEMA_PERMUTATIONS:
    count = 0
    with TRAIN_JSONL.open(encoding="utf-8") as src, AUG_TRAIN_JSONL.open("w", encoding="utf-8") as dst:
        for line in src:
            row = json.loads(line)
            dst.write(json.dumps(row, ensure_ascii=False) + "\n")
            messages = [dict(m) for m in row["messages"]]
            payload = json.loads(messages[1]["content"])
            tools = payload.get("available_tools", [])
            if len(tools) > 1:
                key = f"{SEED}:{row['sample_id']}".encode()
                rng = random.Random(int(hashlib.sha256(key).hexdigest()[:16], 16))
                shuffled = list(tools)
                rng.shuffle(shuffled)
                if shuffled == tools:
                    shuffled = shuffled[1:] + shuffled[:1]
                payload["available_tools"] = shuffled
                messages[1]["content"] = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
                augmented = dict(row)
                augmented["messages"] = messages
                dst.write(json.dumps(augmented, ensure_ascii=False) + "\n")
                count += 1
    TRAIN_FOR_RUN = AUG_TRAIN_JSONL
    print(f"Schema-order variants: {count}; training rows: {sum(1 for _ in AUG_TRAIN_JSONL.open())}")
else:
    TRAIN_FOR_RUN = TRAIN_JSONL


## 5. Preflight rồi resume train

Preflight kiểm tra chiều dài dữ liệu trước khi nạp model train. Cell kế tiếp sẽ tiếp tục từ checkpoint 508; early stopping vẫn bật.

In [ ]:
def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=PROJECT_DIR, check=True)

common = [
    "--train-jsonl", TRAIN_FOR_RUN, "--dev-jsonl", DEV_JSONL,
    "--model-path", MODEL_PATH, "--seed", SEED,
    "--max-sequence-length", MAX_SEQUENCE_LENGTH,
    "--learning-rate", LR,
    "--gradient-accumulation-steps", GRAD_ACCUMULATION,
    "--lora-rank", LORA_RANK, "--lora-alpha", LORA_ALPHA,
    "--lora-dropout", LORA_DROPOUT,
    "--evals-per-epoch", EVALS_PER_EPOCH,
    "--early-stopping-patience", EARLY_STOPPING_PATIENCE,
    "--early-stopping-threshold", EARLY_STOPPING_THRESHOLD,
]
preflight = RUN_DIR / "resume_preflight.json"
run([sys.executable, "training/train_lora.py", *common,
     "--preflight-only", "--auto-context",
     "--preflight-report", preflight,
     "--output-dir", RUN_DIR / "preflight"])
selected_context = json.loads(preflight.read_text())["selected_context"]
common[common.index("--max-sequence-length") + 1] = selected_context
print("Preflight context:", selected_context)

# A true resume must keep the original train/dev data, including schema augmentation.
original_preflight = FULL_DIR / "preflight.json"
if not original_preflight.is_file():
    raise FileNotFoundError("Thiếu full/preflight.json để xác minh dữ liệu resume.")
original = json.loads(original_preflight.read_text())
current = json.loads(preflight.read_text())
for split in ("train", "dev"):
    expected = original["splits"][split]["data_sha256"]
    actual = current["splits"][split]["data_sha256"]
    if actual != expected:
        raise RuntimeError(f"Dữ liệu {split} khác run gốc. Dừng trước khi nạp checkpoint để tránh resume sai.")
print("Train/dev SHA-256 match the original run.")

run([
    sys.executable, "training/train_lora.py", *common,
    "--epochs", MAX_EPOCHS,
    "--output-dir", FULL_DIR,
    "--resume-from-checkpoint", RESUME_CHECKPOINT,
])
if not (FULL_DIR / "final/adapter_config.json").is_file():
    raise FileNotFoundError("Train kết thúc nhưng không thấy full/final adapter.")
print("Resume train hoàn tất:", FULL_DIR / "final")


## 6. Đánh giá toàn bộ dev

Đánh giá adapter mới trên dev. Không tạo public submission trong notebook này.

In [ ]:
DEV_REPORT = RUN_DIR / "dev_resumed_final_full.json"
run([
    sys.executable, "training/evaluate_dev.py",
    "--data-root", DATA_ROOT,
    "--adapter", FULL_DIR / "final",
    "--limit-samples", 0,
    "--report", DEV_REPORT,
])
print("Dev report:", DEV_REPORT)


## Hoàn tất

Adapter sau train nằm ở `lifemate_runs/vast-20261106-124816/full/final/`. Báo cáo dev nằm ở `lifemate_runs/vast-20261106-124816/dev_resumed_final_full.json`. Public inference vẫn tắt; chỉ bật trong notebook inference riêng sau khi xem báo cáo.